# e2m tutorial: TCGA LUAD

This notebook runs the manuscript workflow for one cancer type. It downloads cohort-level Xena STAR counts and MC3 data, evaluates mutation and TMB prediction with held-out folds, trains a reusable multitask model, and interprets one mutation output with two SHAP methods.

The package exposes two equivalent interfaces, and this notebook runs **either** one. Set `USE_OBJECT_API` in the next cell and run top to bottom:

- **Disk / functional flow** (`USE_OBJECT_API = False`) mirrors the command line. Each step writes result files and reads them back. Use it to reproduce the manuscript outputs on disk.
- **Object flow** (`USE_OBJECT_API = True`) keeps everything in memory with a `Dataset` and `E2MModel` / `TmbModel`; methods return pandas directly. Use it for scripting and ad-hoc analysis.

Both flows share the same `DATA_DIR` download cache and both write the trained model to `MODEL_DIR`, so the interpretation (SHAP) and prediction steps are identical either way.

Install with `pip install -e ".[interpretation]"` and run from the repository root. The model and SHAP cells can take substantial time.

In [ ]:
from pathlib import Path
import pandas as pd

import e2m
from e2m import cross_validate, download, embed, explain, head_weights, predict_tmb, train
from e2m import Dataset, E2MModel, TmbModel

e2m.set_verbose()  # print step-by-step progress; call e2m.set_verbose(False) to silence

# Pick one interface and run the notebook top to bottom.
#   False -> disk / functional flow (writes and reads result files, mirrors the CLI)
#   True  -> object flow (Dataset + E2MModel / TmbModel, everything in memory)
USE_OBJECT_API = False

CANCERS = ["LUAD"]
DATA_DIR = Path("e2m_data")
RESULT_DIR = Path("results/luad")
MODEL_DIR = Path("models/luad")
DATA_OPTIONS = {"expression_dataset": "star_counts", "expression_transform": "log1p"}
RESULT_DIR.mkdir(parents=True, exist_ok=True)

### What can these options be?

`e2m.options()` returns the accepted expression datasets, transforms, TMB backends, SHAP methods, and TCGA cancer codes; `e2m.format_options()` prints the same as a readable table (the CLI equivalent is `e2m options`).

In [ ]:
print(e2m.format_options())
# or, programmatically:  e2m.options()["expression_transform"]

## 1. Download and prepare data

The manuscript setup keeps GENCODE v36 protein-coding genes, averages duplicate symbols, retains mutation targets present in at least 5 percent of matched samples, and caps the panel at 400 targets. `Dataset.from_tcga` and `download` share the same cache under `DATA_DIR`, so switching flows does not re-download anything.

In [ ]:
if USE_OBJECT_API:
    data = Dataset.from_tcga(CANCERS, data_dir=DATA_DIR, data_overrides=DATA_OPTIONS)
    expression, mutations = data.expression, data.mutations
else:
    download(
        CANCERS,
        data_dir=DATA_DIR,
        data_overrides=DATA_OPTIONS,
        output_dir=RESULT_DIR / "data",
    )
    expression = pd.read_csv(RESULT_DIR / "data/expression.csv.gz", index_col=0)
    mutations = pd.read_csv(RESULT_DIR / "data/mutations.csv.gz", index_col=0)

expression.shape, mutations.shape, mutations.mean().sort_values(ascending=False).head(10)

## 2. Held-out mutation prediction

Use these out-of-fold results for performance reporting. Normalized AUPRC is 0 at the prevalence baseline and 1 for perfect ranking. The object flow returns the metrics table directly; the disk flow writes `metrics.csv` and reads it back. Both also write the out-of-fold probabilities to `mutation_cv/`.

In [ ]:
if USE_OBJECT_API:
    mutation_metrics = E2MModel().cross_validate(data, output=RESULT_DIR / "mutation_cv")
else:
    cross_validate(
        CANCERS,
        output_dir=RESULT_DIR / "mutation_cv",
        data_dir=DATA_DIR,
        data_overrides=DATA_OPTIONS,
    )
    mutation_metrics = pd.read_csv(RESULT_DIR / "mutation_cv/metrics.csv", index_col=0)

mutation_metrics.sort_values("normalized_auprc", ascending=False).head(15)

## 3. Held-out TMB prediction

Coding MC3 events define TMB. Samples without a coding mutation event are dropped. The default backend, XGBoost, predicts `log2(TMB + 1)`; the object flow accepts other tree backends via `TmbModel(backend=...)`.

In [ ]:
if USE_OBJECT_API:
    tmb_summary = TmbModel().cross_validate(data, output=RESULT_DIR / "tmb_cv")
else:
    predict_tmb(
        CANCERS,
        output_dir=RESULT_DIR / "tmb_cv",
        data_dir=DATA_DIR,
        data_overrides=DATA_OPTIONS,
    )
    tmb_summary = pd.read_csv(RESULT_DIR / "tmb_cv/summary.csv")

tmb_summary

## 4. Train a reusable multitask model

This full-cohort fit is for deployment, embeddings, head weights, and direct neural SHAP. It is not a held-out performance estimate. Both flows write the model bundle to `MODEL_DIR` and leave a fitted `model` object in memory, so every step below is identical regardless of the flow you chose.

In [ ]:
if USE_OBJECT_API:
    model = E2MModel().fit(data)
    model.save(MODEL_DIR)
else:
    train(
        CANCERS,
        output_dir=MODEL_DIR,
        data_dir=DATA_DIR,
        data_overrides=DATA_OPTIONS,
    )
    model = E2MModel.load(MODEL_DIR)

model

## 5. Explain TP53 with manuscript Tree SHAP

This step reads the saved model bundle, so it is the same for both flows. A separate full-cohort XGBoost classifier is used for attribution only; it does not replace the multitask model used for mutation performance. `method` also accepts the other tree backends (`lightgbm`, `random_forest`, `gradient_boosting`); `xgboost` is the manuscript default.

In [ ]:
explain(
    MODEL_DIR,
    target="TP53",
    method="xgboost",
    output_dir=RESULT_DIR / "shap_xgboost_tp53",
    data_dir=DATA_DIR,
)
pd.read_csv(RESULT_DIR / "shap_xgboost_tp53/feature_summary.csv").head(20)

## 6. Explain the multitask TP53 output directly

Gradient SHAP explains the neural probability output using a sampled LUAD background. This inspects the trained network itself; the manuscript interpretation uses the XGBoost method above.

In [ ]:
explain(
    MODEL_DIR,
    target="TP53",
    method="neural",
    output_dir=RESULT_DIR / "shap_neural_tp53",
    data_dir=DATA_DIR,
)
pd.read_csv(RESULT_DIR / "shap_neural_tp53/feature_summary.csv").head(20)

## 7. Export embeddings and output-head weights

The shared encoder gives a 256-dimensional representation per sample; each output head is one weight vector per mutation target.

In [ ]:
if USE_OBJECT_API:
    embeddings = model.embed(data)
    weights = model.head_weights()
    embeddings.to_csv(RESULT_DIR / "sample_embeddings.csv")
    weights.to_csv(RESULT_DIR / "head_weights.csv")
else:
    embed(MODEL_DIR, RESULT_DIR / "data/expression.csv.gz", RESULT_DIR / "sample_embeddings.csv")
    head_weights(MODEL_DIR, RESULT_DIR / "head_weights.csv")
    embeddings = pd.read_csv(RESULT_DIR / "sample_embeddings.csv", index_col=0)
    weights = pd.read_csv(RESULT_DIR / "head_weights.csv", index_col=0)

embeddings.shape, weights.shape

## 8. Predict on held-out or new samples

`model` is a fitted `E2MModel` in both flows, so prediction is the same either way. Pass any samples-by-genes matrix (a held-out slice here, or an external cohort); input genes are aligned to the training features by symbol. See `external_transfer.ipynb` for a full cross-cohort example on an external GEO dataset.

In [ ]:
held_out = expression.tail(8)
model.predict(held_out).iloc[:, :5]

## Interpretation note

SHAP reports features used by a model. In bulk RNA data, those features can reflect mutation-associated expression, subtype, co-mutation, immune cells, stromal cells, or other correlated biology. Do not treat a SHAP association as proof of a direct causal effect.